[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/reinhart-group/generative-copolymer-workshop/blob/main/day2/06_forward_prediction.ipynb)

# Day 2 — Afternoon: RNN Forward Prediction

**Objectives:**
- Understand supervised learning for soft matter: sequence → structure regression
- Learn how RNNs handle discrete polymer sequences
- Load a pre-trained RNN predictor
- Evaluate the model's ability to predict UMAP coordinates ($Z$) from sequences ($X$)

**The big picture.** Notebooks 04 and 05 showed you how to *visualise* polymer structure — turning simulation snapshots into points on a 2-D map. Here we learn the *mapping itself*: we train a recurrent neural network (GRU) to read a binary polymer sequence and predict where it lands on the UMAP map.

This is the **forward model**: given any sequence $X$, predict its structure coordinates $Z = (Z_0, Z_1)$. A well-trained forward model is the core component of inverse design — you can use it to score candidate sequences and search for ones whose predicted $Z$ is close to a target morphology.

In [ ]:
# If running on Colab, uncomment and run this cell first, then restart the runtime:
# !pip install "scikit-learn>=1.3,<2" tqdm
# Note: torch is pre-installed on Colab — do NOT reinstall it here, as that can
# break GPU support or trigger a conflicting runtime restart.

# To verify torch is available:
# import torch; print(torch.__version__, 'CUDA:', torch.cuda.is_available())

**The toolkit.** We will be using:

- `numpy`, `pandas` — data handling
- `matplotlib` — plots
- `torch` (`torch.nn`, `torch.optim`) — model definition and training
- `sklearn` (`train_test_split`, `StandardScaler`) — data splitting and feature normalisation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

## Model and Training Code

The next cell defines the `GRUPredictor` class and the `train_model` helper function. You don't need to modify either — but reading the `GRUPredictor` docstring is worthwhile: it explains why the architecture is **bidirectional** and how the sequence is processed as a time series of monomers.

> **Why define these here?** PyTorch requires a model class to be defined before you can load saved weights into it. Putting the definitions at the top means you can jump straight to Section 2 in future sessions (to load weights and evaluate) without re-running the data-loading cells first.

In [ ]:
# ── Model definition ──────────────────────────────────────────────────────────

class GRUPredictor(nn.Module):
    """
    Bidirectional GRU that reads a binary polymer sequence one monomer at a time
    and predicts the 2-D embedding coordinates (Z0, Z1).

    Architecture
    ------------
    Input  : (batch, 30, 1)  — 30 monomers, each encoded as 0 or 1
    GRU    : bidirectional, so the model reads the chain both left-to-right
             and right-to-left and combines both representations
    Output : (batch, 2)      — predicted Z0 and Z1 coordinates

    Why bidirectional?
    ------------------
    A copolymer with sequence A is structurally equivalent to one read in reverse,
    so we want the model to be equally sensitive to both orientations.
    Bidirectionality gives it both perspectives simultaneously.
    """

    def __init__(self, hidden_dim=64, n_layers=2, dropout=0.1):
        super().__init__()
        self.gru = nn.GRU(
            input_size=1,           # one binary feature per monomer
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        # Concatenate final forward + backward hidden states → predict Z0, Z1
        self.head = nn.Linear(2 * hidden_dim, 2)

    def forward(self, x):
        _, hn = self.gru(x)                      # hn: (n_layers*2, batch, hidden_dim)
        h = torch.cat([hn[-2], hn[-1]], dim=1)   # (batch, 2 * hidden_dim)
        return self.head(h)                       # (batch, 2)


# ── Training loop ─────────────────────────────────────────────────────────────

def train_model(model, X_train_t, Z_train_t, X_test_t, Z_test_t,
                n_epochs=200, batch_size=128, lr=1e-3):
    """
    Train a GRUPredictor using MSE loss on standardized Z coordinates.

    Prints train and test MSE every 20 epochs.
    Returns lists of (train_loss, test_loss) recorded at each print step.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    loader    = DataLoader(TensorDataset(X_train_t, Z_train_t),
                           batch_size=batch_size, shuffle=True)

    train_history, test_history = [], []

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        epoch_loss /= len(loader)

        if (epoch + 1) % 20 == 0:
            model.eval()
            with torch.no_grad():
                test_loss = loss_fn(model(X_test_t), Z_test_t).item()
            train_history.append(epoch_loss)
            test_history.append(test_loss)
            print(f'Epoch {epoch+1:>4}/{n_epochs}  '
                  f'train MSE={epoch_loss:.4f}  test MSE={test_loss:.4f}')

    return train_history, test_history

## 1. Sequence Encoding

A polymer sequence like `'011100101100...'` is already binary — each character is either
`0` (monomer type A) or `1` (monomer type B). To feed it into a neural network, we just
convert each character to an integer.

We then reshape the result to `(n_sequences, 30, 1)` — the GRU will process each of the
30 monomers as one time step, reading the chain from end to end.

We load the UMAP embedding CSV saved in notebook 05 and convert each sequence string into float arrays that PyTorch can consume. We do this in two steps: first load and verify, then encode all sequences and standardise the target coordinates.

**Step 1a — Load the embedding data.** Read the CSV saved by notebook 05. Each row has a binary sequence string and its 2-D UMAP coordinates. We print one example encoding to verify the conversion looks correct before processing the full dataset.

In [ ]:
# --- LOAD DATA ---
EMBEDDING_CSV = 'umap_embedding_for_regression.csv'

embedding_df = pd.read_csv(EMBEDDING_CSV, dtype={'Sequence': str})
sequences    = embedding_df['Sequence'].values
Z            = embedding_df[['Z0', 'Z1']].values.astype(np.float32)

print(f'Loaded {len(sequences)} sequences from {EMBEDDING_CSV}')
print(f'Z range:  Z0 ∈ [{Z[:, 0].min():.2f}, {Z[:, 0].max():.2f}]   '
      f'Z1 ∈ [{Z[:, 1].min():.2f}, {Z[:, 1].max():.2f}]')

# Show one example to verify the encoding
example = sequences[0]
encoded = [int(c) for c in example]
print(f'\nExample sequence : {example}')
print(f'Encoded as ints  : {encoded}')

**Step 1b — Encode and scale.** Convert every sequence string to a `(30, 1)` float array. The extra trailing `1` dimension is required by PyTorch's GRU: it expects `(batch, time_steps, n_features)`, where `n_features=1` because each monomer contributes just one binary value. We also standardise the Z coordinates so both axes have similar magnitude, keeping the MSE loss balanced.

In [ ]:
# Encode all sequences: (n_sequences, 30, 1) — 30 monomers, 1 binary feature each
# The GRU expects input shape (batch, time_steps, features)
X_flat = np.array([[int(c) for c in seq] for seq in sequences], dtype=np.float32)
X      = X_flat.reshape(-1, 30, 1)

# Standardise Z so both coordinates have zero mean and unit variance.
# This makes the MSE loss equally sensitive to errors in Z0 and Z1,
# which can have very different natural scales.
scaler   = StandardScaler()
Z_scaled = scaler.fit_transform(Z)

print(f'X shape: {X.shape}  (n_seqs × 30 monomers × 1 feature)')
print(f'Z shape: {Z_scaled.shape}  (n_seqs × 2 UMAP dims, standardised)')

▶ **What you should see:** the total number of sequences loaded, the Z coordinate ranges, and an example encoding — a list of 30 zeros and ones matching the sequence string character by character. After Step 1b: `X shape: (N, 30, 1)` and `Z shape: (N, 2)`. If the CSV fails to load, check that notebook 05 ran to completion and saved the file.

## 2. RNN Architecture

We use a **Gated Recurrent Unit (GRU)** — a type of recurrent network that maintains a
hidden "memory" as it reads the sequence from left to right (and right to left, since our
model is bidirectional). This is different from treating the whole sequence as a flat input:
the GRU can learn positional effects like "a block of B monomers near the middle matters
differently than one at the end."

The architecture is:

```
monomer₁ → GRU cell → hidden state ─┐
monomer₂ → GRU cell → hidden state  │  → final hidden ─→ Linear → (Z0, Z1)
   ⋮             ⋮                   │
monomer₃₀→ GRU cell → hidden state ─┘
```

(simultaneously in both directions)

**Three things happen in this section:**

1. **Split** the data into train and test sets and convert to PyTorch tensors.
2. **Instantiate** the GRU model and inspect its parameter count.
3. **Load** pre-trained weights if available, or **train** from scratch — then save the weights so you don't need to retrain later.

> **Skip training?** If `gru_predictor.pt` exists in your working directory, Step 2c loads it automatically and skips training. Jump straight to Section 3.

**Step 2a — Train / test split.** Hold out 20% of sequences for evaluation and convert everything to PyTorch tensors. We split the raw sequence strings alongside the arrays so we can identify the hardest-to-predict sequences by name in Section 3.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# 80/20 split — carry raw sequence strings alongside so we can look them up later
X_train, X_test, Z_train, Z_test, seq_train, seq_test = train_test_split(
    X, Z_scaled, sequences, test_size=0.2, random_state=42
)

X_train_t = torch.tensor(X_train).to(device)
X_test_t  = torch.tensor(X_test).to(device)
Z_train_t = torch.tensor(Z_train).to(device)
Z_test_t  = torch.tensor(Z_test).to(device)

print(f'Train: {X_train_t.shape[0]} sequences  |  Test: {X_test_t.shape[0]} sequences')

**Step 2b — Build the model.** Instantiate the GRU predictor and print its layer structure and parameter count. A model with ~40–100k parameters is appropriate for this task: large enough to capture the sequence–structure relationship, small enough to train in a few minutes on CPU.

> **Try it:** change `hidden_dim` to 128 or `n_layers` to 3 for a larger model, or reduce them for a faster one. Re-run Steps 2c and 2d to compare.

In [ ]:
model = GRUPredictor(hidden_dim=64, n_layers=2, dropout=0.1).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'GRUPredictor — {n_params:,} trainable parameters')
print(model)

**Step 2c — Load or train.** If `gru_predictor.pt` exists in the working directory, we load it and skip training. Otherwise we train for 200 epochs and save the weights — so re-running this notebook in future skips straight to evaluation.

In [ ]:
MODEL_WEIGHTS    = 'gru_predictor.pt'
train_history, test_history = None, None

if os.path.exists(MODEL_WEIGHTS):
    model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=device))
    print(f'Loaded pre-trained weights from {MODEL_WEIGHTS}')
    print('Skipping training — jump to Section 3 to evaluate.')
else:
    print('No pre-trained weights found — training from scratch (roughly 1–2 min on CPU)...')
    train_history, test_history = train_model(
        model, X_train_t, Z_train_t, X_test_t, Z_test_t,
        n_epochs=200, batch_size=128, lr=1e-3,
    )
    torch.save(model.state_dict(), MODEL_WEIGHTS)
    print(f'\nWeights saved to {MODEL_WEIGHTS}')

**Step 2d — Learning curve.** If you just trained the model, the plot below shows how MSE evolved over epochs for the train and test sets. A healthy curve shows both lines falling and then flattening, with test MSE staying close to (but slightly above) train MSE.

In [ ]:
# Learning curve — only shown if we just trained the model
if train_history is not None:
    fig, ax = plt.subplots(figsize=(7, 4))
    epochs_shown = range(20, 200 + 1, 20)
    ax.plot(epochs_shown, train_history, marker='o', label='Train MSE')
    ax.plot(epochs_shown, test_history,  marker='o', label='Test MSE')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE (standardised Z)')
    ax.set_title('Learning Curve — GRU Forward Model')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No training history (pre-trained weights were loaded) — skipping learning curve.')

▶ **What you should see:** "Loaded pre-trained weights" (if `gru_predictor.pt` exists) or a stream of epoch lines like `Epoch  20/200  train MSE=0.xxxx  test MSE=0.xxxx` ending with "Weights saved". If training, both MSE values should decrease over epochs. If test MSE rises while train MSE keeps falling, the model is overfitting — try reducing `n_epochs` or increasing `dropout` in Step 2b.

## 3. Evaluation

A good forward model should predict Z coordinates close to the true values for sequences it has never seen (the 20% test set). We assess this with two tools:

1. **Parity plots** — if predictions were perfect, every dot would fall on the red dashed diagonal. Scatter around the diagonal shows prediction error.
2. **RMSE vs baseline** — the model's error compared to a naive baseline that always predicts the mean Z value. If the model is learning, its RMSE should be well below baseline.

We also display the **five worst-predicted test sequences** — sequences whose predicted Z was furthest from the truth. These can reveal whether the model struggles in a particular region of the structure space.

In [ ]:
model.eval()
with torch.no_grad():
    Z_pred_scaled = model(X_test_t).cpu().numpy()

# Unscale predictions and true values back to original UMAP coordinates
Z_pred = scaler.inverse_transform(Z_pred_scaled)
Z_true = scaler.inverse_transform(Z_test)

# --- METRICS ---
rmse_z0 = np.sqrt(np.mean((Z_pred[:, 0] - Z_true[:, 0]) ** 2))
rmse_z1 = np.sqrt(np.mean((Z_pred[:, 1] - Z_true[:, 1]) ** 2))
base_z0 = np.sqrt(np.mean((Z_true[:, 0] - Z_true[:, 0].mean()) ** 2))
base_z1 = np.sqrt(np.mean((Z_true[:, 1] - Z_true[:, 1].mean()) ** 2))

print('           RMSE       Baseline')
print(f'  Z0  :   {rmse_z0:.4f}     {base_z0:.4f}')
print(f'  Z1  :   {rmse_z1:.4f}     {base_z1:.4f}')
print(f'  Relative improvement — Z0: {(1 - rmse_z0/base_z0)*100:.1f}%   '
      f'Z1: {(1 - rmse_z1/base_z1)*100:.1f}%')

# --- PARITY PLOTS ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, (name, rmse, baseline) in enumerate(
        zip(['Z0', 'Z1'], [rmse_z0, rmse_z1], [base_z0, base_z1])):
    ax = axes[i]
    ax.scatter(Z_true[:, i], Z_pred[:, i], s=6, alpha=0.3, color='steelblue')
    lo = min(Z_true[:, i].min(), Z_pred[:, i].min()) - 0.1
    hi = max(Z_true[:, i].max(), Z_pred[:, i].max()) + 0.1
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1.2, label='Perfect prediction')
    ax.set_xlabel(f'True {name}')
    ax.set_ylabel(f'Predicted {name}')
    ax.set_title(f'{name}   RMSE={rmse:.4f}  (baseline {baseline:.4f})')
    ax.legend(fontsize=8)
plt.suptitle('GRU Forward Model — Sequence → Structure Coordinates', fontsize=13)
plt.tight_layout()
plt.show()

# --- FIVE WORST PREDICTIONS ---
errors    = np.sqrt(((Z_pred - Z_true) ** 2).sum(axis=1))
worst_idx = np.argsort(errors)[-5:][::-1]
print('\nFive worst-predicted test sequences:')
print(f'  {"Sequence":<32} {"True Z0":>8} {"True Z1":>8} {"Pred Z0":>8} {"Pred Z1":>8} {"Error":>7}')
for idx in worst_idx:
    print(f'  {seq_test[idx]:<32} '
          f'{Z_true[idx, 0]:>+8.3f} {Z_true[idx, 1]:>+8.3f} '
          f'{Z_pred[idx, 0]:>+8.3f} {Z_pred[idx, 1]:>+8.3f} '
          f'{errors[idx]:>7.3f}')

▶ **What you should see:** RMSE values for Z0 and Z1 that are substantially below the baseline (ideally 30–60% improvement), parity plots with points clustered near the red diagonal, and a table of five sequences with the largest prediction errors showing their actual binary strings. If RMSE is close to the baseline, the model likely needs more training epochs — increase `n_epochs` in Step 2c, delete `gru_predictor.pt`, and re-run.